In [1]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser
import os
from dotenv import load_dotenv
import re
from typing import List, Dict, Optional
import json

load_dotenv()

ModuleNotFoundError: No module named 'marker'

In [52]:
raw_config = {
    "output_format": "markdown",
    "use_llm": True,
}

config_parser = ConfigParser(raw_config)

converter = PdfConverter(
    config = config_parser.generate_config_dict(),
    artifact_dict = create_model_dict(),
    processor_list = config_parser.get_processors(),
    renderer = config_parser.get_renderer(),
    llm_service = config_parser.get_llm_service(),
)

rendered = converter("./data/cdc.pdf")

Loaded layout model s3://layout/2025_02_18 on device cpu with dtype torch.float32
Loaded texify model s3://texify/2025_02_18 on device cpu with dtype torch.float32
Loaded recognition model s3://text_recognition/2025_02_18 on device cpu with dtype torch.float32
Loaded table recognition model s3://table_recognition/2025_02_18 on device cpu with dtype torch.float32
Loaded detection model s3://text_detection/2025_02_28 on device cpu with dtype torch.float32
Loaded detection model s3://inline_math_detection/2025_02_24 on device cpu with dtype torch.float32


Recognizing layout: 100%|██████████| 1/1 [00:17<00:00, 17.11s/it]
LLM layout relabelling: 2it [00:00,  2.10it/s]


400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API Key not found. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API Key not found. Please pass a valid API key.'}]}}
400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API Key not found. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API Key not found. Please pass a valid API key.'}]}}


Running OCR Error Detection: 100%|██████████| 1/1 [00:01<00:00,  1.48s/it]
Detecting bboxes: 0it [00:00, ?it/s]
Detecting bboxes: 0it [00:00, ?it/s]
LLMTableMergeProcessor running: 0it [00:00, ?it/s]
LLM processors running: 0it [00:00, ?it/s]


In [53]:

text, _, images = text_from_rendered(rendered)
if images:
    for path, image in images.items():
        print(f"Image {path}: {image}")
        
        # Save the image
        try:
            os.makedirs('./output', exist_ok=True)
            image.save('./output/'+path, image.format)
            print(f"Image saved to ./data/{path}")
        except Exception as e:
            print(f"Error displaying/saving image: {e}")
else:
    print("No images found in the PDF.")

# Print the extracted text
print("\nExtracted text (including image descriptions):")
print(text)

No images found in the PDF.

Extracted text (including image descriptions):
# **APPEL D'OFFRES RELATIF A**

## **« Modernisation des services des œuvres universitaires»**

## **CAHIER DES CHARGES**

## **REF : N° 09/2025**

## **PREAMBULE**

Le ministère de l'enseignement supérieur et de la recherche scientifique accorde un intérêt accru aux conditions d'étude et de vie des étudiants et souhaite lancer son projet stratégique d'amélioration de l'écosystème des études et de la qualité de vie estudiantine à travers la réforme du système des œuvres universitaires. L'objectif est d'offrir un cadre de vie propice à l'épanouissement académique et personnel de la force active de demain : les étudiants.

Confronté au problème de massification estudiantine, le ministère fait face à un véritable défi pour assurer aux étudiants les meilleures conditions de vie possible. En effet, les problèmes d'orientation des étudiants, de surcharge des résidences universitaires, etc. rendent le suivi et l'évalu

In [ ]:
def find_all_sentence_contexts(
    full_text: str,
    search_terms: List[str],
    language: str = 'fr'
) -> Dict[str, List[Dict[str, Optional[str]]]]:
    
    # Basic sentence splitting regex (works for French and English)
    sentence_endings = r'(?<!\w\.\w.)(?<![A-ZÀ-Ü][a-zà-ü]\.)(?<=\.|\?|\!|\…|\n)\s+'
    sentences = [s.strip() for s in re.split(sentence_endings, full_text) if s.strip()]
    
    # Precompile case-insensitive regex patterns for each term
    patterns = {
        term: re.compile(rf'(?<!\w){re.escape(term)}(?!\w)', re.IGNORECASE)
        for term in search_terms
    }
    
    results = {term: [] for term in search_terms}
    
    for idx, sentence in enumerate(sentences):
        for term, pattern in patterns.items():
            if pattern.search(sentence):
                context = {
                    'prev': sentences[idx-1] if idx > 0 else None,
                    'match': sentence,
                    'next': sentences[idx+1] if idx < len(sentences)-1 else None
                }
                results[term].append(context)
    
    return results

In [116]:
print (contexts)

{'prestataire': [{'prev': "L'accompagnement de ces acteurs dans cette nouvelle mission est primordial pour la réussite de ce projet.", 'match': 'Le prestataire est engagé pour fournir, sur appel, une étude basée sur des technologies de pointe et des normes efficaces et universelles.', 'next': "L'offre doit couvrir principalement les volets suivants :"}, {'prev': '# **Description du projet**', 'match': "Le présent cahier des charges définit les conditions et les modalités selon lesquelles le prestataire s'engage pour fournir, sur appel, des solutions pour répondre aux trois volets décrits ci-après.", 'next': 'La durée prévue du projet est de **12 mois**.'}, {'prev': "Cette annexe a pour objectif de définir le format type de la réponse à l'appel d'offres.", 'match': "Le prestataire est dans l'obligation de suivre les instructions décrites ci-après.", 'next': 'Les offres techniques et financières doivent être élaborées selon les instructions décrites.'}, {'prev': 'Introduction\n- 2.', 'ma

In [ ]:
terms = ["prestataire", "contrat"] 
contexts = find_all_sentence_contexts(text, terms)

In [126]:
with open('./output/test.json', 'w', encoding='utf-8') as f:
        json.dump(contexts, f, ensure_ascii=False, indent=2)
print(f"Contexts saved")


Contexts saved


In [127]:
for term, occurrences in contexts.items():
    print(f"\n{'='*80}")
    print(f"Term: '{term}' - Found {len(occurrences)} occurrence(s)")
    print(f"{'='*80}")
    
    for i, ctx in enumerate(occurrences, 1):
        print(f"\nContext {i}:")
        if ctx['prev']:
            print(f"[Previous] {ctx['prev']}")
        print(f"[Match]    {ctx['match']}")
        if ctx['next']:
            print(f"[Next]     {ctx['next']}")
        print("-" * 40)


Term: 'prestataire' - Found 5 occurrence(s)

Context 1:
[Previous] L'accompagnement de ces acteurs dans cette nouvelle mission est primordial pour la réussite de ce projet.
[Match]    Le prestataire est engagé pour fournir, sur appel, une étude basée sur des technologies de pointe et des normes efficaces et universelles.
[Next]     L'offre doit couvrir principalement les volets suivants :
----------------------------------------

Context 2:
[Previous] # **Description du projet**
[Match]    Le présent cahier des charges définit les conditions et les modalités selon lesquelles le prestataire s'engage pour fournir, sur appel, des solutions pour répondre aux trois volets décrits ci-après.
[Next]     La durée prévue du projet est de **12 mois**.
----------------------------------------

Context 3:
[Previous] Cette annexe a pour objectif de définir le format type de la réponse à l'appel d'offres.
[Match]    Le prestataire est dans l'obligation de suivre les instructions décrites ci-après.
[

In [129]:
# To convert PDF to images


# pip install pymupdf

# import fitz

# pdffile = "scanned.pdf"
# doc = fitz.open(pdffile)
# zoom = 4
# mat = fitz.Matrix(zoom, zoom)
# count = 0
# # Count variable is to get the number of pages in the pdf
# for p in doc:
#     count += 1
# for i in range(count):
#     val = f"image_{i+1}.png"
#     page = doc.load_page(i)
#     pix = page.get_pixmap(matrix=mat)
#     pix.save(val)
# doc.close()result